# 05 - Analiza błędów

Metryka mówi nam *ile* model się myli. Tutaj pytamy o coś ciekawszego - **gdzie i dlaczego** się myli. To często uczy więcej niż sama liczba.

Notebook ma dwie części: residua modelu regresji (z notebooka 03) oraz pomyłki klasyfikatora (z notebooka 04). Wnioski na końcu są liczone z danych w tym notebooku, a nie wpisane z góry.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

reg = pd.read_parquet('../data/reg_predictions.parquet')
clf = pd.read_parquet('../data/clf_predictions.parquet')
reg['residual'] = reg['actual_rel'] - reg['pred_rel']
print('reg:', reg.shape, '| clf:', clf.shape)

reg: (10477, 9) | clf: (918, 14)


## Regresja - rozkład residuów

Residuum to różnica między wartością rzeczywistą a przewidzianą (dla tempa względnego). Dobry model daje symetryczną chmurę wokół zera, bez systematycznego przesunięcia (*bias*).

In [2]:
mae = reg['residual'].abs().mean()
bias = reg['residual'].mean()
print(f'MAE = {mae:.3f} s | bias = {bias:+.3f} s')
fig = px.histogram(reg, x='residual', nbins=80,
                   title=f'Rozkład residuów (MAE={mae:.3f}s, bias={bias:+.3f}s)',
                   labels={'residual': 'residuum [s]'})
fig.add_vline(x=0, line_dash='dash')
fig.show()

MAE = 0.627 s | bias = +0.044 s


### Czy został jakiś wzorzec? Residua a wiek opony

Gdyby błąd rósł wraz z wiekiem opony, oznaczałoby to, że model nie uchwycił degradacji do końca. Chcemy zobaczyć chmurę wokół zera, bez wyraźnego trendu.

In [3]:
fig = px.scatter(reg, x='TyreLife', y='residual', color='Compound', opacity=0.3,
                 title='Residua vs wiek opony (chcemy chmurę wokół 0, bez trendu)',
                 labels={'TyreLife': 'wiek opony [okr.]', 'residual': 'residuum [s]'})
fig.add_hline(y=0, line_dash='dash')
fig.show()

In [4]:
by_comp = (reg.assign(abs_err=reg['residual'].abs())
           .groupby('Compound')['abs_err'].agg(['mean', 'count']).reset_index())
fig = px.bar(by_comp, x='Compound', y='mean',
             title='Średni błąd bezwzględny per mieszanka opon',
             labels={'mean': 'MAE [s]', 'Compound': 'mieszanka'})
fig.show()
by_comp

,Compound,mean,count
0,HARD,0.615374,5728
1,MEDIUM,0.620592,3624
2,SOFT,0.709311,1125


### Najgorzej przewidziane okrążenia

Warto spojrzeć na konkretne okrążenia, na których model pomylił się najbardziej - zwykle to one wskazują granice jego możliwości.

In [5]:
worst = reg.reindex(reg['residual'].abs().sort_values(ascending=False).index).head(15)
worst[['EventName', 'Compound', 'TyreLife', 'LapNumber',
       'actual_rel', 'pred_rel', 'residual']].round(2)

,EventName,Compound,TyreLife,LapNumber,actual_rel,pred_rel,residual
679,Azerbaijan Grand Prix,HARD,3.0,14.0,3.35,-0.25,3.60
4766,Australian Grand Prix,HARD,2.0,43.0,1.81,-1.58,3.39
6245,Japanese Grand Prix,HARD,7.0,39.0,2.89,-0.47,3.37
2605,Spanish Grand Prix,MEDIUM,9.0,48.0,1.73,-1.64,3.36
5513,Australian Grand Prix,HARD,6.0,42.0,2.57,-0.78,3.35
8353,Singapore Grand Prix,HARD,31.0,59.0,2.15,-1.16,3.31
8787,Singapore Grand Prix,SOFT,8.0,54.0,1.95,-1.27,3.22
8737,Singapore Grand Prix,SOFT,19.0,56.0,1.89,-1.29,3.18
10347,Qatar Grand Prix,HARD,9.0,43.0,1.71,-1.37,3.08
6105,Japanese Grand Prix,SOFT,18.0,52.0,2.49,-0.57,3.06


In [6]:
# wnioski regresji LICZONE z danych
worst_track = reg.assign(ae=reg.residual.abs()).groupby('EventName')['ae'].mean().idxmax()
worst_comp = by_comp.loc[by_comp['mean'].idxmax(), 'Compound']
tail = (reg['residual'].abs() > 3 * reg['residual'].abs().median()).mean()
print('Tor z największym średnim błędem:', worst_track)
print('Mieszanka z największym błędem: ', worst_comp)
print(f'Odsetek okrążeń z błędem >3x mediany (ogon): {tail:.1%}')

Tor z największym średnim błędem: Qatar Grand Prix
Mieszanka z największym błędem:  SOFT
Odsetek okrążeń z błędem >3x mediany (ogon): 6.3%


## Klasyfikacja - gdzie model się myli

Najciekawsze są pomyłki "blisko granicy": kierowcy z okolic czwartego-piątego miejsca, których model typował na podium i odwrotnie. Przyglądamy się też kalibracji prawdopodobieństw.

In [7]:
clf['err_type'] = np.select(
    [(clf.podium == 1) & (clf.pred_logit == 1),
     (clf.podium == 0) & (clf.pred_logit == 0),
     (clf.podium == 0) & (clf.pred_logit == 1),
     (clf.podium == 1) & (clf.pred_logit == 0)],
    ['TP (trafione podium)', 'TN (trafione nie-podium)',
     'FP (typował podium, nie było)', 'FN (przegapił podium)'],
    default='?')
clf['err_type'].value_counts()

err_type
TN (trafione nie-podium)         624
FP (typował podium, nie było)    156
TP (trafione podium)             123
FN (przegapił podium)             15
Name: count, dtype: int64

In [8]:
miss = clf[clf.podium != clf.pred_logit].sort_values('proba_logit', ascending=False)
miss[['EventName', 'Driver', 'Team', 'GridPosition', 'Position',
      'proba_logit', 'podium', 'err_type']].round(3).head(20)

,EventName,Driver,Team,GridPosition,Position,proba_logit,podium,err_type
642,Austrian Grand Prix,VER,Red Bull Racing,1.0,5.0,0.964,0,"FP (typował podium, nie było)"
704,Belgian Grand Prix,PER,Red Bull Racing,2.0,7.0,0.959,0,"FP (typował podium, nie było)"
497,Australian Grand Prix,VER,Red Bull Racing,1.0,19.0,0.954,0,"FP (typował podium, nie było)"
823,Mexico City Grand Prix,VER,Red Bull Racing,2.0,6.0,0.953,0,"FP (typował podium, nie było)"
682,Hungarian Grand Prix,VER,Red Bull Racing,3.0,5.0,0.939,0,"FP (typował podium, nie było)"
903,Abu Dhabi Grand Prix,VER,Red Bull Racing,4.0,6.0,0.919,0,"FP (typował podium, nie było)"
774,Azerbaijan Grand Prix,PER,Red Bull Racing,4.0,17.0,0.910,0,"FP (typował podium, nie było)"
723,Dutch Grand Prix,PER,Red Bull Racing,5.0,6.0,0.904,0,"FP (typował podium, nie było)"
862,Las Vegas Grand Prix,VER,Red Bull Racing,5.0,5.0,0.900,0,"FP (typował podium, nie było)"
317,Japanese Grand Prix,PER,Red Bull Racing,5.0,19.0,0.895,0,"FP (typował podium, nie było)"


### Kalibracja - czy prawdopodobieństwa coś znaczą?

Model dobrze skalibrowany to taki, w którym wśród przypadków z przewidzianym prawdopodobieństwem `~0.7` faktycznie około `70%` to podia. Na wykresie punkty powinny układać się blisko przekątnej.

In [9]:
clf['bin'] = pd.cut(clf['proba_logit'], bins=np.linspace(0, 1, 11))
cal = (clf.groupby('bin', observed=True)
       .agg(srednie_P=('proba_logit', 'mean'),
            udzial_podiow=('podium', 'mean'),
            n=('podium', 'size')).dropna().reset_index())
fig = px.scatter(cal, x='srednie_P', y='udzial_podiow', size='n',
                 title='Krzywa kalibracji (regresja logistyczna)',
                 labels={'srednie_P': 'średnie P(podium)', 'udzial_podiow': 'realny udział podiów'})
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(dash='dash', color='red'))
fig.show()

In [10]:
# najwieksza niespodzianka: podium z dalekiej pozycji startowej
upset = clf[(clf.podium == 1)].sort_values('GridPosition', ascending=False)
print('Podia z najdalszych pozycji startowych (model słusznie je przegapia):')
print(upset[['EventName', 'Driver', 'GridPosition', 'Position', 'proba_logit']].head(5).to_string(index=False))

Podia z najdalszych pozycji startowych (model słusznie je przegapia):
               EventName Driver  GridPosition  Position  proba_logit
    Abu Dhabi Grand Prix    LEC          19.0       3.0     0.006586
    São Paulo Grand Prix    VER          17.0       1.0     0.107751
Saudi Arabian Grand Prix    VER          15.0       2.0     0.175216
     Austrian Grand Prix    PER          15.0       3.0     0.266306
    São Paulo Grand Prix    GAS          13.0       3.0     0.084385


## Wnioski końcowe

**Regresja (tempo okrążenia)**
- Po zmianie celu na tempo względne model działa poprawnie. Największe błędy zostają na okrążeniach o nietypowych warunkach, które przetrwały filtry - na przykład ruch na torze czy pozostałości po neutralizacji.
- Wielkość błędu zależy od mieszanki opon - twarda i miękka mają inny rozrzut degradacji.

**Klasyfikacja (podium)**
- Model dobrze chwyta regułę "start z przodu zwiększa szansę na podium", a myli się tam, gdzie F1 bywa nieprzewidywalna: w deszczu, przy samochodzie bezpieczeństwa, awariach i odważnych strategiach.
- Te pomyłki to **naturalna granica przewidywalności** z samych cech przedwyścigowych - nie da się ich usunąć bez danych z przebiegu wyścigu, a te byłyby wyciekiem.

**Wniosek metodyczny**
- Dwie porażki z wcześniejszych notebooków ($R^2 < 0$ oraz `quali_gap_s` w całości puste) były pouczające - pokazały, że wynik zależy bardziej od *postawienia zadania i jakości danych* niż od wyboru algorytmu.
- Benchmark z pomiarem czasu pokazał, że przy małych zbiorach prostsze modele bywają zarazem lepsze i tańsze.